# Main Figure 3 - Lung fibrosis COSTE/TRU analysis

            This notebook is a cleaned, English-commented manuscript code file.

            **Provenance.** `Vannan_2023_Lung_Fibrosis/Rcode/sfplotR/example/plot.R`, regional Structure Score script, and TRU remodelling activity script.

            The notebook keeps the original manuscript data paths where they were found. If a path points to
            `/data/taobo.hu` or `/mnt/taobo.hu`, run the notebook on the A100 server or adapt the path to the
            matching mounted Y-drive location.


In [ ]:
# Panel mapping:
# - Figure 3a-d, shared setup. Loads the lung fibrosis Seurat object, sfplotR functions, cohort sample IDs, and output paths.
#

library(Seurat)
library(dplyr)
library(tidyr)
library(ggplot2)

PROJECT_DIR <- "Y:/long/publication_datasets/Vannan_2023_Lung_Fibrosis"
SFPLOTR_DIR <- file.path(PROJECT_DIR, "Rcode/sfplotR")
OUTPUT_DIR <- "outputs/main_figure_3_lung_fibrosis_TRU"
dir.create(OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)

source(file.path(SFPLOTR_DIR, "R/compute_cophenetic_distances_from_df.R"))
source(file.path(SFPLOTR_DIR, "R/plot_cophenetic_heatmap.R"))
seurat_data <- readRDS(file.path(SFPLOTR_DIR, "example/GSE250346_Seurat_GSE250346_CORRECTED_SEE_RDS_README_082024.rds"))
la_samples <- c("TILD080LA", "TILD028LA", "TILD167LA", "TILD113LA", "TILD111LA", "VUILD49LA", "TILD130LA", "VUILD110LA", "TILD117LA", "VUILD78LA", "VUILD91LA", "VUILD48LA2", "VUILD102LA", "VUILD96LA", "VUILD48LA1")
ma_samples <- c("TILD315MA", "VUILD58MA", "TILD049MA", "TILD299MA", "TILD117MA2", "VUILD141MA", "VUILD142MA", "VUILD106MA", "VUILD115MA", "TILD117MA1", "TILD175MA", "VUILD78MA", "VUILD91MA", "VUILD104MA1", "VUILD105MA2", "VUILD102MA", "VUILD107MA", "VUILD96MA", "VUILD104MA2", "VUILD105MA1")


In [ ]:
# Panel mapping:
# - Figure 3a-b. Computes sample-level Cell-GPS/COSTE SSS values for LA and MA lung fibrosis samples; the exported table is the source for remodeling panels.
#

# Compute all-sample COSTE/SSS values for respiratory remodeling panels.
all_results <- list()
for (sample_id in c(la_samples, ma_samples)) {
  submeta <- seurat_data@meta.data %>%
    filter(sample == sample_id) %>%
    transmute(x = x_centroid, y = y_centroid, group = final_CT)
  if (nrow(submeta) < 100) next
  result <- compute_cophenetic_distances_from_df(submeta, cluster_col = "group", x_col = "x", y_col = "y", method = "average")
  df <- result$row_cophenetic_df %>%
    as.data.frame() %>%
    tibble::rownames_to_column("row") %>%
    pivot_longer(-row, names_to = "column", values_to = "value") %>%
    mutate(sample = sample_id, group = ifelse(sample_id %in% la_samples, "LA", "MA"))
  all_results[[sample_id]] <- df
}
final_df <- bind_rows(all_results)
write.csv(final_df, file.path(OUTPUT_DIR, "cophenetic_distances_searcher_D_score_in_all_samples.csv"), row.names = FALSE)


In [ ]:
# Panel mapping:
# - Figure 3c-d. Plots the AT1/AT2/Capillary remodeling SSS comparison and writes the regional TRS source table used for downstream figure assembly.
#

# Plot the AT1/AT2/Capillary SSS panel and export the regional TRS source table.
valid_levels <- c("AT1", "AT2", "Capillary")
rrs_df <- final_df %>% filter(row %in% valid_levels, column %in% valid_levels, row != column)
p_rrs <- ggplot(rrs_df, aes(x = group, y = value, fill = group)) +
  geom_boxplot(outlier.shape = NA, alpha = 0.75) +
  geom_jitter(width = 0.15, size = 0.8, alpha = 0.6) +
  facet_grid(row ~ column) +
  theme_bw(base_size = 10) +
  labs(x = NULL, y = "COSTE SSS", title = "AT1/AT2/Capillary spatial remodeling")
ggsave(file.path(OUTPUT_DIR, "Figure_3_AT1_AT2_Capillary_SSS.pdf"), p_rrs, width = 8, height = 6, useDingbats = FALSE)
